In [ ]:
# MovieLens

import random
import math
import numpy as np
import pandas as pd
from collections import defaultdict
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# File paths
# ------------------------------------------------------------
# These CSV files are expected to contain:
# - ratings: user-item interactions with timestamps
# - users: user side features
# - movies: item side features
# ============================================================
RATINGS_PATH = "ratings.csv"
USERS_PATH = "users.csv"
MOVIES_PATH = "movies.csv"

# ============================================================
# Global hyperparameters and experiment settings
# ============================================================

SEED = 42
# Fixed random seed for reproducibility.
# 42 is a very common default choice: it has no special mathematical meaning here,
# but keeping the seed fixed helps make data split, negative sampling,
# and model initialization repeatable.

BATCH_SIZE = 2048
# Large batch size to improve training throughput on dense tensor operations.
# Since each positive sample expands into multiple candidate items
# (1 positive + several negatives), the effective number of scored pairs is large.
# 2048 is a practical compromise between speed and memory usage.

EMBED_DIM = 128
# Dimensionality of the final user/item latent representation.
# 128 is large enough to capture richer preference patterns than very small embeddings,
# while still being computationally manageable.

HIDDEN_DIM = 128
# Hidden size of the MLP in both user and item towers.
# Keeping it equal to EMBED_DIM is a simple balanced design:
# enough capacity without making the network unnecessarily large.

DROPOUT = 0.2
# Dropout helps regularize the MLP and reduce overfitting.
# 0.2 is a moderate value: strong enough to help generalization,
# but not so strong that useful signal is heavily destroyed.

LR = 1e-3
# Adam usually works well around 1e-3 as a starting learning rate.
# This is a standard stable choice for recommendation models like this.

WEIGHT_DECAY = 1e-6
# A very light L2 regularization on weights.
# Small because embeddings already carry lots of sparse information,
# and too much regularization may underfit.

NUM_EPOCHS = 20
# Maximum number of training epochs.
# Training may stop earlier because early stopping is enabled.

PATIENCE = 3
# Early stopping patience:
# stop if validation NDCG does not improve for 3 consecutive epochs.
# This helps prevent overfitting and saves training time.

NUM_NEG_TRAIN = 8
# For each positive training instance, sample 8 negatives.
# This is a common pointwise ranking setup:
# enough negatives to provide contrast, without making each batch too expensive.

NUM_NEG_EVAL = 99
# During evaluation, use the classical leave-one-out style:
# 1 positive item vs 99 negatives.
# This makes HR@K / NDCG@K comparable to many recommendation papers.

NEG_FROM_EXPLICIT_PROB = 0.3
# Probability of sampling a negative from items that the user explicitly rated low.
# The idea is to mix:
# - "harder" negatives from known disliked items
# - random unseen negatives for broader coverage
# 0.3 means 30% explicit negatives, 70% random negatives.

POS_THRESHOLD = 4
# Ratings >= 4 are treated as positive feedback.
# This is a common binarization rule in MovieLens-like datasets:
# 4 and 5 typically indicate liking, while 1-3 are neutral/negative.

TOPK = 10
# Evaluate recommendation quality at top-10,
# which is a common and practical ranking cutoff.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Use GPU if available; otherwise fall back to CPU.


def set_seed(seed=42):
    """
    Set random seeds across Python, NumPy, and PyTorch.

    Why this matters:
    Recommendation experiments can vary due to randomness in:
    - parameter initialization
    - data shuffling
    - negative sampling

    Fixing all seeds makes results more reproducible.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# 1. Load data
# ============================================================
ratings = pd.read_csv(RATINGS_PATH)
users = pd.read_csv(USERS_PATH)
movies = pd.read_csv(MOVIES_PATH)

print("ratings:", ratings.shape)
print("users:", users.shape)
print("movies:", movies.shape)

# Check that the necessary columns exist before proceeding.
# This fails early if the dataset format is not as expected.
required_rating_cols = {"userId", "movieId", "rating", "timestamp"}
required_user_cols = {"userId", "gender", "age", "occupation"}
required_movie_cols = {"movieId", "genres"}

assert required_rating_cols.issubset(ratings.columns), "ratings.csv Missing necessary columns"
assert required_user_cols.issubset(users.columns), "users.csv Missing necessary columns"
assert required_movie_cols.issubset(movies.columns), "movies.csv Missing necessary columns"

# Sort interactions by user and time so that later chronological splitting is valid.
ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)


# ============================================================
# 2. Per-user chronological train/val/test split
# ------------------------------------------------------------
# For each user:
# - first 80% interactions -> train
# - next 10% -> validation
# - last 10% -> test
#
# This is better than random splitting for recommendation because it
# respects temporal order and more closely matches real-world prediction:
# use past behavior to predict future behavior.
# ============================================================
train_parts, val_parts, test_parts = [], [], []

for user_id, user_df in ratings.groupby("userId"):
    user_df = user_df.sort_values("timestamp")
    n = len(user_df)

    # Users with fewer than 3 interactions cannot be reasonably split into
    # train/val/test while keeping at least one interaction per split.
    if n < 3:
        continue

    train_end = int(n * 0.8)
    val_end = int(n * 0.9)

    # Safety adjustments to ensure:
    # - at least 1 interaction in train
    # - validation is after train
    # - test remains non-empty
    train_end = max(train_end, 1)
    if val_end <= train_end:
        val_end = train_end + 1
    if val_end >= n:
        val_end = n - 1

    train_parts.append(user_df.iloc[:train_end])
    val_parts.append(user_df.iloc[train_end:val_end])
    test_parts.append(user_df.iloc[val_end:])

train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)


def is_positive(rating):
    """
    Convert an explicit numeric rating into a binary label.

    Positive = rating >= POS_THRESHOLD
    Negative = rating < POS_THRESHOLD

    This converts the recommendation problem from explicit rating prediction
    into implicit-style binary preference learning.
    """
    return 1 if rating >= POS_THRESHOLD else 0


# Add binary labels to all splits.
for df in [train_df, val_df, test_df]:
    df["label"] = df["rating"].apply(is_positive)

print("\ntrain label counts:")
print(train_df["label"].value_counts())
print("\nval label counts:")
print(val_df["label"].value_counts())
print("\ntest label counts:")
print(test_df["label"].value_counts())


# ============================================================
# 3. Build ID encoders from training data only
# ------------------------------------------------------------
# Important design choice:
# we only create user/item index mappings from the training set.
#
# Why?
# This avoids leaking future validation/test-only IDs into training.
# Any user/item not present in training cannot be embedded reliably anyway.
# ============================================================
train_user_ids = sorted(train_df["userId"].unique())
train_item_ids = sorted(train_df["movieId"].unique())

user2idx = {u: i for i, u in enumerate(train_user_ids)}
item2idx = {m: i for i, m in enumerate(train_item_ids)}

num_users = len(user2idx)
num_items = len(item2idx)

print("\nnum_users:", num_users)
print("num_items:", num_items)


def encode_df(df, user2idx, item2idx):
    """
    Filter a dataframe so that only train-known users/items remain,
    then map raw IDs to contiguous integer indices.

    Why contiguous indices?
    PyTorch embedding layers require integer indices in [0, num_embeddings-1].
    """
    df = df.copy()

    # Drop rows whose user/item did not appear in training.
    # This avoids unseen-ID lookup errors in embedding layers.
    df = df[df["userId"].isin(user2idx)]
    df = df[df["movieId"].isin(item2idx)]

    df["user_idx"] = df["userId"].map(user2idx)
    df["item_idx"] = df["movieId"].map(item2idx)
    return df.reset_index(drop=True)


train_df = encode_df(train_df, user2idx, item2idx)
val_df = encode_df(val_df, user2idx, item2idx)
test_df = encode_df(test_df, user2idx, item2idx)

# Separate positive and negative interactions for later use.
train_pos_df = train_df[train_df["label"] == 1].copy().reset_index(drop=True)
train_neg_df = train_df[train_df["label"] == 0].copy().reset_index(drop=True)
val_pos_df = val_df[val_df["label"] == 1].copy().reset_index(drop=True)
test_pos_df = test_df[test_df["label"] == 1].copy().reset_index(drop=True)

print("\ntrain_pos:", train_pos_df.shape)
print("train_neg:", train_neg_df.shape)
print("val_pos:", val_pos_df.shape)
print("test_pos:", test_pos_df.shape)


# ============================================================
# 4. Encode user side features
# ------------------------------------------------------------
# We use:
# - gender
# - age
# - occupation
#
# These features are turned into categorical indices and later embedded.
# ============================================================
users_train = users[users["userId"].isin(train_user_ids)].copy()
users_train["user_idx"] = users_train["userId"].map(user2idx)

gender_values = sorted(users_train["gender"].astype(str).unique())
age_values = sorted(users_train["age"].astype(int).unique())
occ_values = sorted(users_train["occupation"].astype(int).unique())

gender2idx = {g: i for i, g in enumerate(gender_values)}
age2idx = {a: i for i, a in enumerate(age_values)}
occ2idx = {o: i for i, o in enumerate(occ_values)}

num_genders = len(gender2idx)
num_ages = len(age2idx)
num_occs = len(occ2idx)

# Precompute one categorical feature index per user.
# Arrays are aligned with user_idx, so lookup is O(1) later.
user_gender_idx = np.zeros(num_users, dtype=np.int64)
user_age_idx = np.zeros(num_users, dtype=np.int64)
user_occ_idx = np.zeros(num_users, dtype=np.int64)

for row in users_train.itertuples():
    u = row.user_idx
    user_gender_idx[u] = gender2idx[str(row.gender)]
    user_age_idx[u] = age2idx[int(row.age)]
    user_occ_idx[u] = occ2idx[int(row.occupation)]


# ============================================================
# 5. Encode item side features (genres)
# ------------------------------------------------------------
# Movie genres are multi-hot encoded:
# e.g. "Action|Adventure|Sci-Fi" becomes a sparse binary vector.
# ============================================================
movies_train = movies[movies["movieId"].isin(train_item_ids)].copy()
movies_train["item_idx"] = movies_train["movieId"].map(item2idx)

# Build genre vocabulary from training items only.
all_genres = set()
for g in movies_train["genres"].fillna("(no genres listed)").astype(str):
    for token in g.split("|"):
        all_genres.add(token)

genre_vocab = sorted(all_genres)
genre2idx = {g: i for i, g in enumerate(genre_vocab)}
num_genres = len(genre2idx)

# Multi-hot matrix of shape [num_items, num_genres].
item_genre_matrix = np.zeros((num_items, num_genres), dtype=np.float32)

for row in movies_train.itertuples():
    item_idx = row.item_idx
    genres_str = str(row.genres)
    for g in genres_str.split("|"):
        item_genre_matrix[item_idx, genre2idx[g]] = 1.0


# ============================================================
# 6. Build interaction lookup structures
# ------------------------------------------------------------
# These dictionaries support:
# - negative sampling
# - filtering already seen items during evaluation
# ============================================================
train_user_seen_items = defaultdict(set)
for row in train_df.itertuples():
    train_user_seen_items[row.user_idx].add(row.item_idx)

train_user_pos_items = defaultdict(set)
for row in train_pos_df.itertuples():
    train_user_pos_items[row.user_idx].add(row.item_idx)

train_user_explicit_neg_items = defaultdict(list)
for row in train_neg_df.itertuples():
    train_user_explicit_neg_items[row.user_idx].append(row.item_idx)

# For validation, only training history is considered "already seen".
val_seen_user_items = defaultdict(set)
for row in train_df.itertuples():
    val_seen_user_items[row.user_idx].add(row.item_idx)

# For test, both train and validation history are considered seen,
# because test prediction happens after both.
test_seen_user_items = defaultdict(set)
for row in train_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)
for row in val_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)


def build_leave_one_out_positive(pos_df):
    """
    For each user, keep the last positive item in this split
    as the leave-one-out evaluation target.

    Why this design?
    In top-K recommendation, a standard protocol is:
    - take one ground-truth positive item
    - rank it against many negatives
    - check whether the model places it near the top

    Since the data is already time-sorted, the "last" positive interaction
    naturally serves as the future target for that split.
    """
    user_one_pos = {}
    for row in pos_df.itertuples():
        user_one_pos[row.user_idx] = row.item_idx
    return user_one_pos


val_user_one_pos = build_leave_one_out_positive(val_pos_df)
test_user_one_pos = build_leave_one_out_positive(test_pos_df)

val_eval_users = list(val_user_one_pos.keys())
test_eval_users = list(test_user_one_pos.keys())

print("\nval_eval_users:", len(val_eval_users))
print("test_eval_users:", len(test_eval_users))


# ============================================================
# 7. Training dataset with dynamic negative sampling
# ------------------------------------------------------------
# Each training sample is built from:
# - 1 positive item
# - N sampled negative items
#
# This is a pointwise formulation because each candidate item receives
# an independent binary label (1 or 0), and BCE loss is applied.
# ============================================================
class PointwiseRecDataset(Dataset):
    def __init__(
        self,
        pos_df,
        user_pos_items,
        user_explicit_neg_items,
        num_items,
        num_neg=8,
        neg_from_explicit_prob=0.3
    ):
        """
        Args:
            pos_df:
                Dataframe containing only positive user-item interactions.
                Each row becomes one training anchor.
            user_pos_items:
                Dict[user] -> set of items the user positively interacted with.
                Used to avoid sampling obvious false negatives.
            user_explicit_neg_items:
                Dict[user] -> list of explicitly disliked/low-rated items.
                Used to sample harder negatives sometimes.
            num_items:
                Total number of items in training vocabulary.
            num_neg:
                Number of negatives sampled per positive.
            neg_from_explicit_prob:
                Probability of sampling from explicit negatives instead of random items.
        """
        self.users = pos_df["user_idx"].to_numpy()
        self.pos_items = pos_df["item_idx"].to_numpy()
        self.user_pos_items = user_pos_items
        self.user_explicit_neg_items = user_explicit_neg_items
        self.num_items = num_items
        self.num_neg = num_neg
        self.neg_from_explicit_prob = neg_from_explicit_prob

    def __len__(self):
        return len(self.users)

    def sample_one_negative(self, user):
        """
        Sample one negative item for a user.

        Strategy:
        1. With some probability, draw from items the user explicitly rated low.
           These are informative "hard" negatives.
        2. Otherwise, draw a random item that is not in the user's positive set.

        Note:
        We only ensure the sampled item is not a known positive.
        This is common in implicit recommendation training, although it means
        some random negatives may actually be unknown positives.
        """
        explicit_negs = self.user_explicit_neg_items[user]

        if len(explicit_negs) > 0 and random.random() < self.neg_from_explicit_prob:
            return random.choice(explicit_negs)

        while True:
            neg_item = random.randint(0, self.num_items - 1)
            if neg_item not in self.user_pos_items[user]:
                return neg_item

    def __getitem__(self, idx):
        """
        Return one training sample:
        - a user
        - a list of candidate items [1 positive + num_neg negatives]
        - corresponding binary labels [1, 0, 0, ..., 0]
        """
        user = int(self.users[idx])
        pos_item = int(self.pos_items[idx])

        # Use a set to avoid duplicate negatives within the same sample.
        neg_items = set()
        while len(neg_items) < self.num_neg:
            neg_item = self.sample_one_negative(user)
            if neg_item != pos_item:
                neg_items.add(neg_item)

        neg_items = list(neg_items)

        items = [pos_item] + neg_items
        labels = [1.0] + [0.0] * self.num_neg

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(items, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float)
        )


train_dataset = PointwiseRecDataset(
    pos_df=train_pos_df,
    user_pos_items=train_user_pos_items,
    user_explicit_neg_items=train_user_explicit_neg_items,
    num_items=num_items,
    num_neg=NUM_NEG_TRAIN,
    neg_from_explicit_prob=NEG_FROM_EXPLICIT_PROB
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)
# num_workers=0 keeps data loading simple and stable across environments,
# especially in notebooks or Windows systems where multiprocessing can be tricky.


# ============================================================
# 8. Two-tower recommendation model with side features
# ------------------------------------------------------------
# User tower:
#   user ID embedding + gender embedding + age embedding + occupation embedding
#
# Item tower:
#   item ID embedding + genre feature projection
#
# Final score:
#   dot(user_vector, item_vector)
#
# This architecture is common in retrieval/recommendation because:
# - user and item are encoded separately
# - representation learning is efficient
# - dot product provides a simple similarity score
# ============================================================
class TwoTowerSideFeatureModel(nn.Module):
    def __init__(
        self,
        num_users,
        num_items,
        num_genders,
        num_ages,
        num_occs,
        num_genres,
        user_gender_idx,
        user_age_idx,
        user_occ_idx,
        item_genre_matrix,
        embed_dim=128,
        hidden_dim=128,
        dropout=0.2
    ):
        super().__init__()

        # ----------------------------
        # User-side embeddings
        # ----------------------------
        self.user_id_emb = nn.Embedding(num_users, embed_dim)

        # Side-feature embedding sizes are kept small (8).
        # Reason:
        # these features are low-cardinality categorical metadata,
        # so they usually do not need large embeddings.
        self.gender_emb = nn.Embedding(num_genders, 8)
        self.age_emb = nn.Embedding(num_ages, 8)
        self.occ_emb = nn.Embedding(num_occs, 8)

        # User tower input dimension:
        # user_id_emb + gender_emb + age_emb + occ_emb
        user_input_dim = embed_dim + 8 + 8 + 8

        # MLP fuses user ID and user side features into one final user vector.
        self.user_mlp = nn.Sequential(
            nn.Linear(user_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        # ----------------------------
        # Item-side embeddings
        # ----------------------------
        self.item_id_emb = nn.Embedding(num_items, embed_dim)

        # Convert multi-hot genre vector into a dense 16-dim genre representation.
        # 16 is small but enough for compressed genre semantics.
        self.genre_linear = nn.Linear(num_genres, 16)

        item_input_dim = embed_dim + 16

        # MLP fuses item ID signal and genre signal.
        self.item_mlp = nn.Sequential(
            nn.Linear(item_input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        # register_buffer stores non-trainable tensors inside the module.
        # They move automatically with model.to(device), but are not optimized.
        self.register_buffer("user_gender_idx", torch.tensor(user_gender_idx, dtype=torch.long))
        self.register_buffer("user_age_idx", torch.tensor(user_age_idx, dtype=torch.long))
        self.register_buffer("user_occ_idx", torch.tensor(user_occ_idx, dtype=torch.long))
        self.register_buffer("item_genre_matrix", torch.tensor(item_genre_matrix, dtype=torch.float))

        self._init_weights()

    def _init_weights(self):
        """
        Initialize parameters.

        Embeddings:
            small normal initialization (std=0.01)
            common for embedding tables to start with small values.

        Linear layers:
            Xavier uniform initialization
            suitable for feed-forward layers with ReLU-like activations.
        """
        for emb in [self.user_id_emb, self.gender_emb, self.age_emb, self.occ_emb, self.item_id_emb]:
            nn.init.normal_(emb.weight, std=0.01)

        for module in list(self.user_mlp) + list(self.item_mlp):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

        nn.init.xavier_uniform_(self.genre_linear.weight)
        nn.init.zeros_(self.genre_linear.bias)

    def encode_user(self, user_idx):
        """
        Encode users into dense vectors.

        user_idx: shape [B]

        Steps:
        1. look up user ID embedding
        2. look up side-feature embeddings via precomputed feature-index buffers
        3. concatenate all user-related signals
        4. feed into MLP for fusion
        """
        user_id_vec = self.user_id_emb(user_idx)
        gender_vec = self.gender_emb(self.user_gender_idx[user_idx])
        age_vec = self.age_emb(self.user_age_idx[user_idx])
        occ_vec = self.occ_emb(self.user_occ_idx[user_idx])

        user_input = torch.cat([user_id_vec, gender_vec, age_vec, occ_vec], dim=-1)
        user_vec = self.user_mlp(user_input)
        return user_vec

    def encode_item(self, item_idx):
        """
        Encode items into dense vectors.

        item_idx: shape [B]

        Steps:
        1. look up item ID embedding
        2. fetch multi-hot genre features
        3. normalize by number of genres
        4. linearly project genres to a dense vector
        5. concatenate ID and genre signals
        6. fuse with MLP

        Why divide by genre_count?
        Without normalization, movies with many genres would produce larger
        raw feature magnitudes than movies with only one genre.
        Averaging helps make genre contribution more comparable across items.
        """
        item_id_vec = self.item_id_emb(item_idx)

        genre_feats = self.item_genre_matrix[item_idx]
        genre_count = genre_feats.sum(dim=-1, keepdim=True).clamp(min=1.0)
        genre_vec = self.genre_linear(genre_feats / genre_count)

        item_input = torch.cat([item_id_vec, genre_vec], dim=-1)
        item_vec = self.item_mlp(item_input)
        return item_vec

    def forward(self, user_idx, item_idx):
        """
        Score user-item pairs with dot product similarity.

        Why dot product?
        - simple
        - efficient
        - standard in two-tower retrieval models
        - naturally measures alignment between user and item representations
        """
        user_vec = self.encode_user(user_idx)
        item_vec = self.encode_item(item_idx)
        scores = (user_vec * item_vec).sum(dim=-1)
        return scores


model = TwoTowerSideFeatureModel(
    num_users=num_users,
    num_items=num_items,
    num_genders=num_genders,
    num_ages=num_ages,
    num_occs=num_occs,
    num_genres=num_genres,
    user_gender_idx=user_gender_idx,
    user_age_idx=user_age_idx,
    user_occ_idx=user_occ_idx,
    item_genre_matrix=item_genre_matrix,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)


# ============================================================
# 9. Loss and optimizer
# ============================================================
criterion = nn.BCEWithLogitsLoss()
# BCEWithLogitsLoss combines sigmoid + binary cross-entropy in a numerically stable way.
# Since model outputs raw scores (logits), this is the correct choice.

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
# Adam is a solid default optimizer for sparse-ish embedding + MLP training.


def train_one_epoch_pointwise(model, dataloader, optimizer, criterion, device):
    """
    Train the model for one epoch using pointwise binary classification.

    Each batch contains:
    - user: shape [B]
    - items: shape [B, 1 + num_neg]
    - labels: shape [B, 1 + num_neg]

    We flatten all user-item candidates into a single dimension so that the model
    scores every candidate independently, then apply BCE loss against binary labels.
    """
    model.train()
    total_loss = 0.0
    total_examples = 0

    for user, items, labels in dataloader:
        user = user.to(device)
        items = items.to(device)
        labels = labels.to(device)

        batch_size, num_candidates = items.shape

        # Expand each user index to align with all its candidate items.
        # Example:
        # user shape [B] -> [B, num_candidates] -> flattened [B * num_candidates]
        user_flat = user.unsqueeze(1).expand(-1, num_candidates).reshape(-1)
        item_flat = items.reshape(-1)
        label_flat = labels.reshape(-1)

        optimizer.zero_grad()
        logits = model(user_flat, item_flat)
        loss = criterion(logits, label_flat)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * label_flat.size(0)
        total_examples += label_flat.size(0)

    return total_loss / total_examples


def evaluate_1vs99(
    model,
    eval_users,
    user_one_pos,
    seen_user_items,
    num_items,
    k=10,
    num_neg=99,
    device="cpu",
    seed=42
):
    """
    Evaluate ranking quality with 1 positive item vs 99 negatives.

    For each evaluation user:
    - take the leave-one-out positive item
    - sample 99 negatives not already seen
    - score 100 candidates
    - rank them
    - compute HR@K, NDCG@K, and AP (here effectively MAP over one positive)

    Metrics:
    - HR@K:
        whether the positive item appears in top-K
    - NDCG@K:
        rank-sensitive reward; higher when the positive appears nearer the top
    - MAP:
        with one relevant item, AP = 1 / rank if found, else 0
    """
    model.eval()
    rng = np.random.default_rng(seed)

    hr_list = []
    ndcg_list = []
    map_list = []

    with torch.no_grad():
        for user in eval_users:
            pos_item = user_one_pos[user]
            seen_items = set(seen_user_items[user])

            # If the target positive item is in "seen", remove it because it is the item
            # we are trying to rank among negatives.
            if pos_item in seen_items:
                seen_items.remove(pos_item)

            # Sample unseen negatives.
            neg_items = set()
            while len(neg_items) < num_neg:
                neg = int(rng.integers(0, num_items))
                if neg != pos_item and neg not in seen_items:
                    neg_items.add(neg)

            candidates = [pos_item] + list(neg_items)

            user_tensor = torch.full((len(candidates),), user, dtype=torch.long, device=device)
            item_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model(user_tensor, item_tensor).cpu().numpy()

            # Rank candidates by descending predicted score.
            ranked_idx = np.argsort(-scores)
            ranked_items = [candidates[i] for i in ranked_idx]

            hr = 1.0 if pos_item in ranked_items[:k] else 0.0

            if pos_item in ranked_items[:k]:
                rank = ranked_items.index(pos_item)

                # NDCG for a single relevant item:
                # DCG = 1 / log2(rank + 2), IDCG = 1
                ndcg = 1.0 / math.log2(rank + 2)

                # AP for one relevant item is reciprocal rank in this setup.
                ap = 1.0 / (rank + 1)
            else:
                ndcg = 0.0
                ap = 0.0

            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(ap)

    if len(hr_list) == 0:
        return 0.0, 0.0, 0.0

    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))


# ============================================================
# 10. Training loop with early stopping
# ------------------------------------------------------------
# We monitor validation NDCG@10 and keep the best model state.
# NDCG is a strong ranking metric because it rewards putting the true positive
# higher in the list, not just somewhere in the top-K.
# ============================================================
best_val_ndcg = -1.0
best_state = None
best_epoch = -1
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_pointwise(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )

    val_hr10, val_ndcg10, val_map10 = evaluate_1vs99(
        model=model,
        eval_users=val_eval_users,
        user_one_pos=val_user_one_pos,
        seen_user_items=val_seen_user_items,
        num_items=num_items,
        k=TOPK,
        num_neg=NUM_NEG_EVAL,
        device=DEVICE,
        seed=SEED
    )

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val HR@{TOPK}: {val_hr10:.4f} | "
        f"Val NDCG@{TOPK}: {val_ndcg10:.4f} | "
        f"Val MAP@{TOPK}: {val_map10:.4f}"
    )

    # Save best checkpoint based on validation NDCG.
    if val_ndcg10 > best_val_ndcg:
        best_val_ndcg = val_ndcg10
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
        print("Saved best model.")
    else:
        patience_counter += 1

    # Stop training if validation performance has not improved for PATIENCE epochs.
    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break


# Restore best model before final test evaluation.
if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Val NDCG@{TOPK}: {best_val_ndcg:.4f}")


# ============================================================
# 11. Final test evaluation
# ------------------------------------------------------------
# Use the best validation checkpoint to report final test performance.
# ============================================================
test_hr10, test_ndcg10, test_map10 = evaluate_1vs99(
    model=model,
    eval_users=test_eval_users,
    user_one_pos=test_user_one_pos,
    seen_user_items=test_seen_user_items,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

print("\nFinal Test Results:")
print(f"Test HR@{TOPK}: {test_hr10:.4f}")
print(f"Test NDCG@{TOPK}: {test_ndcg10:.4f}")
print(f"Test MAP@{TOPK}: {test_map10:.4f}")

ratings: (1000209, 4)
users: (6040, 5)
movies: (3883, 3)
train: (797758, 4)
val: (99692, 4)
test: (102759, 4)

train label counts:
label
1    469223
0    328535
Name: count, dtype: int64

val label counts:
label
1    53003
0    46689
Name: count, dtype: int64

test label counts:
label
1    53055
0    49704
Name: count, dtype: int64

num_users: 6040
num_items: 3666

train_pos: (469223, 7)
train_neg: (328535, 7)
val_pos: (52997, 7)
test_pos: (53042, 7)

val_eval_users: 5787
test_eval_users: 5824
Epoch 01/20 | Train Loss: 0.3049 | Val HR@10: 0.4794 | Val NDCG@10: 0.2576 | Val MAP@10: 0.1906
Saved best model.
Epoch 02/20 | Train Loss: 0.2682 | Val HR@10: 0.5383 | Val NDCG@10: 0.2914 | Val MAP@10: 0.2168
Saved best model.
Epoch 03/20 | Train Loss: 0.2550 | Val HR@10: 0.5711 | Val NDCG@10: 0.3101 | Val MAP@10: 0.2309
Saved best model.
Epoch 04/20 | Train Loss: 0.2461 | Val HR@10: 0.6036 | Val NDCG@10: 0.3321 | Val MAP@10: 0.2494
Saved best model.
Epoch 05/20 | Train Loss: 0.2398 | Val HR@10:

In [ ]:
#lastfm

import random
import math
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# Data source
# ============================================================
# The Last.fm interaction file contains implicit feedback:
# - userID   : user identifier
# - artistID : item identifier (artist)
# - weight   : interaction strength / listening frequency
#
# Unlike explicit rating datasets, this dataset does not directly say
# whether a user "likes" or "dislikes" an item. We only know that
# a larger weight usually suggests stronger preference or stronger exposure.
DATA_PATH = "user_artists.dat"

# ============================================================
# Training / model hyperparameters
# ============================================================
SEED = 42

# A relatively large batch size improves training throughput
# and is usually fine for embedding-based recommendation models.
# 2048 is often a practical trade-off between speed and memory usage.
BATCH_SIZE = 2048

# Dimensionality of user/item ID embeddings.
# 128 is a common middle-ground:
# - expressive enough to capture latent preference signals
# - not too large to overfit or consume too much memory
EMBED_DIM = 128

# Hidden size in the MLP layers of each tower.
# Here it is larger than EMBED_DIM to give the projection MLP
# more capacity before compressing back to the final embedding space.
HIDDEN_DIM = 256

# Dropout for regularization.
# 0.1 is a light regularization choice:
# - enough to reduce overfitting a bit
# - unlikely to destabilize training
DROPOUT = 0.1

# Adam learning rate.
# 5e-4 is slightly more conservative than 1e-3, which is helpful
# when training ranking models with many sampled negatives.
LR = 5e-4

# Small L2 regularization.
# Kept intentionally small because embedding models are sensitive to
# over-regularization, especially when the dataset is sparse.
WEIGHT_DECAY = 1e-6

# Maximum training epochs.
# Early stopping will usually stop training before reaching this limit
# if validation performance stops improving.
NUM_EPOCHS = 20

# Stop if validation metric does not improve for 3 consecutive epochs.
# This helps prevent overfitting and saves compute time.
PATIENCE = 3

# Number of negative samples per positive instance during training.
# 30 is fairly large for implicit recommendation and helps the model
# learn stronger discrimination between positive and non-observed items.
NUM_NEG_TRAIN = 30

# Number of negative samples during evaluation.
# "1 positive vs 99 negatives" is a very common offline ranking protocol.
NUM_NEG_EVAL = 99

# Top-K cutoff used in HR@K / NDCG@K / MAP@K.
TOPK = 10

# Minimum number of interactions required for a user/item to remain.
# This filtering reduces extreme sparsity and helps the model learn
# more reliable embeddings for both users and artists.
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5

# Probability of sampling a negative item from the popularity distribution
# instead of uniform random sampling.
# Popular items are often harder negatives because users are more likely
# to have been exposed to them, so they are more informative for training.
POPULAR_NEG_PROB = 0.3

# Use GPU if available, otherwise CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed=42):
    """
    Fix random seeds for reproducibility.

    This affects:
    - Python random
    - NumPy random
    - PyTorch random
    - CUDA random states

    In recommendation experiments, this is especially important because:
    - negative sampling is stochastic
    - embedding initialization is stochastic
    - dataloader shuffling is stochastic
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


# ============================================================
# Load raw interaction data
# ============================================================
# The file is tab-separated, so sep="\t" is required.
df = pd.read_csv(DATA_PATH, sep="\t")

required_cols = {"userID", "artistID", "weight"}
missing_cols = required_cols - set(df.columns)
if missing_cols:
    raise ValueError(f"Last.fm file Missing necessary columns: {missing_cols}")

print("Raw shape:", df.shape)
print(df.head())


# ============================================================
# Basic filtering for data quality
# ============================================================
# Count how many interactions each user and item has.
user_counts = df["userID"].value_counts()
item_counts = df["artistID"].value_counts()

# Keep only:
# - users with at least MIN_USER_INTERACTIONS interactions
# - items with at least MIN_ITEM_INTERACTIONS interactions
#
# Why do this?
# Extremely cold users/items are hard to model reliably.
# Filtering them reduces noise and makes offline evaluation more stable.
df = df[
    df["userID"].isin(user_counts[user_counts >= MIN_USER_INTERACTIONS].index) &
    df["artistID"].isin(item_counts[item_counts >= MIN_ITEM_INTERACTIONS].index)
].copy()

print("\nAfter filtering:", df.shape)


# ============================================================
# Build a pseudo time order
# ============================================================
# The Last.fm file here does not provide a real timestamp.
# To still perform leave-one-out style splitting, we create a pseudo order.
#
# We sort interactions within each user by:
# 1. weight
# 2. artistID
#
# Then assign cumcount()+1 as pseudo_ts.
#
# Important note:
# This is NOT true chronological order.
# It is only a deterministic ordering strategy so that train/val/test
# can be split consistently.
df = df.sort_values(["userID", "weight", "artistID"], ascending=[True, True, True]).reset_index(drop=True)
df["pseudo_ts"] = df.groupby("userID").cumcount() + 1

print("\nWith pseudo order:")
print(df.head(10))


# ============================================================
# Leave-last-two-out split per user
# ============================================================
# For each user:
# - all but last 2 interactions -> training
# - second last interaction     -> validation
# - last interaction            -> test
#
# This is a common leave-one-out / leave-two-out evaluation style in
# recommendation tasks, especially for implicit feedback.
train_parts, val_parts, test_parts = [], [], []

for user_id, user_df in df.groupby("userID"):
    user_df = user_df.sort_values("pseudo_ts")
    n = len(user_df)

    # If fewer than 3 interactions remain, we cannot form train/val/test.
    if n < 3:
        continue

    train_parts.append(user_df.iloc[:-2])
    val_parts.append(user_df.iloc[-2:-1])
    test_parts.append(user_df.iloc[-1:])

train_df = pd.concat(train_parts).reset_index(drop=True)
val_df = pd.concat(val_parts).reset_index(drop=True)
test_df = pd.concat(test_parts).reset_index(drop=True)

print("\nTrain:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

# In implicit recommendation, observed interactions are usually treated
# as positive signals. There are no explicit negative labels in the raw data.
train_df["label"] = 1
val_df["label"] = 1
test_df["label"] = 1


# ============================================================
# Build index mappings from training data only
# ============================================================
# We only use IDs present in the training set to define the embedding tables.
# This avoids leakage and keeps the setup consistent with real model training.
train_user_ids = sorted(train_df["userID"].unique())
train_item_ids = sorted(train_df["artistID"].unique())

user2idx = {u: i for i, u in enumerate(train_user_ids)}
item2idx = {m: i for i, m in enumerate(train_item_ids)}

num_users = len(user2idx)
num_items = len(item2idx)

print("\nnum_users:", num_users)
print("num_items:", num_items)


def encode_df(df, user2idx, item2idx):
    """
    Convert raw user/item IDs into contiguous integer indices.

    Why this is needed:
    PyTorch Embedding layers require integer indices in [0, N-1].

    We also drop rows whose users/items are not present in training,
    because the model has no learned embeddings for unseen IDs.
    """
    df = df.copy()
    df = df[df["userID"].isin(user2idx)]
    df = df[df["artistID"].isin(item2idx)]
    df["user_idx"] = df["userID"].map(user2idx)
    df["item_idx"] = df["artistID"].map(item2idx)
    return df.reset_index(drop=True)

train_df = encode_df(train_df, user2idx, item2idx)
val_df = encode_df(val_df, user2idx, item2idx)
test_df = encode_df(test_df, user2idx, item2idx)

print("\nEncoded train:", train_df.shape)
print("Encoded val:", val_df.shape)
print("Encoded test:", test_df.shape)


# ============================================================
# Build user history lookup tables
# ============================================================
# These structures are used for:
# - excluding already seen items in evaluation
# - avoiding invalid negatives during sampling
train_user_seen_items = defaultdict(set)
for row in train_df.itertuples():
    train_user_seen_items[row.user_idx].add(row.item_idx)

# During validation, items seen in training should not be counted as candidates.
val_seen_user_items = defaultdict(set)
for row in train_df.itertuples():
    val_seen_user_items[row.user_idx].add(row.item_idx)

# During test, items seen in both training and validation should be excluded.
test_seen_user_items = defaultdict(set)
for row in train_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)
for row in val_df.itertuples():
    test_seen_user_items[row.user_idx].add(row.item_idx)


def build_leave_one_out_positive(df):
    """
    For each user, keep the last positive item in this split
    as the leave-one-out evaluation target.

    Since validation and test each contain one held-out interaction per user
    in this split design, this function effectively maps:
        user -> held-out positive item
    """
    user_one_pos = {}
    for row in df.sort_values(["user_idx", "pseudo_ts"]).itertuples():
        user_one_pos[row.user_idx] = row.item_idx
    return user_one_pos

val_user_one_pos = build_leave_one_out_positive(val_df)
test_user_one_pos = build_leave_one_out_positive(test_df)

val_eval_users = list(val_user_one_pos.keys())
test_eval_users = list(test_user_one_pos.keys())

print("\nval_eval_users:", len(val_eval_users))
print("test_eval_users:", len(test_eval_users))


# ============================================================
# Build popularity-based negative sampling distribution
# ============================================================
# Popular items are more likely to be sampled as negatives with some probability.
# This is useful because:
# - very unpopular random items are often too easy as negatives
# - popular items tend to be stronger / harder negatives
item_popularity = train_df["item_idx"].value_counts().sort_index()

# Start with ones to avoid zero-probability items.
pop_array = np.ones(num_items, dtype=np.float64)

for item_idx, cnt in item_popularity.items():
    pop_array[item_idx] = float(cnt)

# Raise counts to the 0.75 power.
# This is a common smoothing trick:
# - keeps popular items more likely
# - but avoids making the sampling distribution too dominated by the top few items
pop_probs = np.power(pop_array, 0.75)
pop_probs = pop_probs / pop_probs.sum()


class PointwiseImplicitDataset(Dataset):
    """
    Pointwise training dataset for implicit recommendation.

    Each training example contains:
    - 1 observed positive item
    - N sampled negative items

    Returned structure:
    - user   : scalar user index
    - items  : [pos_item, neg_1, neg_2, ..., neg_N]
    - labels : [1, 0, 0, ..., 0]

    Why pointwise instead of pairwise?
    - simpler implementation
    - stable optimization with BCEWithLogitsLoss
    - easy to batch efficiently
    """

    def __init__(
        self,
        pos_df,
        user_seen_items,
        num_items,
        num_neg=20,
        pop_probs=None,
        popular_neg_prob=0.3
    ):
        self.users = pos_df["user_idx"].to_numpy()
        self.pos_items = pos_df["item_idx"].to_numpy()
        self.user_seen_items = user_seen_items
        self.num_items = num_items
        self.num_neg = num_neg
        self.pop_probs = pop_probs
        self.popular_neg_prob = popular_neg_prob

    def __len__(self):
        return len(self.users)

    def sample_one_negative(self, user):
        """
        Sample one negative item for a user.

        Strategy:
        - with probability 'popular_neg_prob', sample according to item popularity
        - otherwise sample uniformly at random

        Why mix both strategies?
        - uniform negatives increase coverage of the whole item space
        - popularity-based negatives are often harder and more realistic
        """
        while True:
            if self.pop_probs is not None and random.random() < self.popular_neg_prob:
                neg_item = int(np.random.choice(self.num_items, p=self.pop_probs))
            else:
                neg_item = random.randint(0, self.num_items - 1)

            # Do not sample an item the user has already interacted with.
            if neg_item not in self.user_seen_items[user]:
                return neg_item

    def __getitem__(self, idx):
        user = int(self.users[idx])
        pos_item = int(self.pos_items[idx])

        # Use a set so negatives within one sample are unique.
        neg_items = set()
        while len(neg_items) < self.num_neg:
            neg_item = self.sample_one_negative(user)
            if neg_item != pos_item:
                neg_items.add(neg_item)

        neg_items = list(neg_items)

        items = [pos_item] + neg_items
        labels = [1.0] + [0.0] * self.num_neg

        return (
            torch.tensor(user, dtype=torch.long),
            torch.tensor(items, dtype=torch.long),
            torch.tensor(labels, dtype=torch.float)
        )


train_dataset = PointwiseImplicitDataset(
    pos_df=train_df,
    user_seen_items=train_user_seen_items,
    num_items=num_items,
    num_neg=NUM_NEG_TRAIN,
    pop_probs=pop_probs,
    popular_neg_prob=POPULAR_NEG_PROB
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

sample_batch = next(iter(train_loader))
print("\nSample batch shapes:")
print("user:", sample_batch[0].shape)
print("items:", sample_batch[1].shape)
print("labels:", sample_batch[2].shape)


class TwoTowerImplicitModel(nn.Module):
    """
    Basic two-tower model for implicit recommendation.

    Architecture:
    - user tower: user ID embedding -> MLP -> final user vector
    - item tower: item ID embedding -> MLP -> final item vector
    - final score: dot product of user and item vectors

    Why a two-tower design?
    - user and item are encoded independently
    - efficient for retrieval and ranking
    - easy to scale to large candidate sets
    - conceptually simple and strong as a baseline
    """

    def __init__(self, num_users, num_items, embed_dim=128, hidden_dim=256, dropout=0.2):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)

        # User tower:
        # first project the raw embedding to a higher hidden space,
        # then map back to the final embedding space.
        self.user_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        # Item tower mirrors the user tower.
        self.item_mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim)
        )

        self._init_weights()

    def _init_weights(self):
        """
        Initialize trainable parameters.

        Embeddings:
        - small Gaussian initialization is a standard choice

        Linear layers:
        - Xavier uniform works well for MLP layers with ReLU activation
        """
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

        for module in list(self.user_mlp) + list(self.item_mlp):
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def encode_user(self, user_idx):
        """
        Convert user indices into dense user representations.
        """
        user_vec = self.user_mlp(self.user_emb(user_idx))
        return user_vec

    def encode_item(self, item_idx):
        """
        Convert item indices into dense item representations.
        """
        item_vec = self.item_mlp(self.item_emb(item_idx))
        return item_vec

    def forward(self, user_idx, item_idx):
        """
        Compute relevance scores for user-item pairs.

        The dot product is used because it naturally measures alignment
        between the user and item representations in the same latent space.
        """
        user_vec = self.encode_user(user_idx)
        item_vec = self.encode_item(item_idx)
        scores = (user_vec * item_vec).sum(dim=-1)
        return scores


model = TwoTowerImplicitModel(
    num_users=num_users,
    num_items=num_items,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(DEVICE)

print("\nModel:")
print(model)


# BCEWithLogitsLoss is used because:
# - we are doing binary classification over sampled candidates
# - it combines sigmoid + BCE in a numerically stable form
criterion = nn.BCEWithLogitsLoss()

# Adam is a strong default optimizer for neural recommendation models.
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def train_one_epoch_pointwise(model, dataloader, optimizer, criterion, device):
    """
    Train the model for one epoch using pointwise BCE loss.

    Input batch format:
    - user   : [B]
    - items  : [B, C]
    - labels : [B, C]

    Since the model expects pairwise inputs of shape [N],
    we flatten the batch from [B, C] into [B*C].
    """
    model.train()
    total_loss = 0.0
    total_examples = 0

    for user, items, labels in dataloader:
        user = user.to(device)
        items = items.to(device)
        labels = labels.to(device)

        batch_size, num_candidates = items.shape

        # Repeat each user index across all candidate items.
        user_flat = user.unsqueeze(1).expand(-1, num_candidates).reshape(-1)
        item_flat = items.reshape(-1)
        label_flat = labels.reshape(-1)

        optimizer.zero_grad()

        logits = model(user_flat, item_flat)
        loss = criterion(logits, label_flat)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * label_flat.size(0)
        total_examples += label_flat.size(0)

    return total_loss / total_examples


def evaluate_1vs99(
    model,
    eval_users,
    user_one_pos,
    seen_user_items,
    num_items,
    k=10,
    num_neg=99,
    device="cpu",
    seed=42
):
    """
    Evaluate ranking quality using the classic 1-vs-99 protocol.

    For each evaluation user:
    1. take one held-out positive item
    2. sample 99 unseen negative items
    3. score all 100 candidate items
    4. rank them by predicted score
    5. compute HR@K, NDCG@K, MAP@K

    Why this protocol?
    - much cheaper than ranking against the full catalog
    - widely used in recommendation benchmarks
    - gives a simple and comparable estimate of ranking quality
    """
    model.eval()
    rng = np.random.default_rng(seed)

    hr_list = []
    ndcg_list = []
    map_list = []

    with torch.no_grad():
        for user in eval_users:
            pos_item = user_one_pos[user]
            seen_items = set(seen_user_items[user])

            # Remove the target item if it is in the seen set;
            # otherwise it would be excluded from the candidate pool.
            if pos_item in seen_items:
                seen_items.remove(pos_item)

            neg_items = set()
            while len(neg_items) < num_neg:
                neg = int(rng.integers(0, num_items))
                if neg != pos_item and neg not in seen_items:
                    neg_items.add(neg)

            candidates = [pos_item] + list(neg_items)

            user_tensor = torch.full((len(candidates),), user, dtype=torch.long, device=device)
            item_tensor = torch.tensor(candidates, dtype=torch.long, device=device)

            scores = model(user_tensor, item_tensor).cpu().numpy()
            ranked_idx = np.argsort(-scores)
            ranked_items = [candidates[i] for i in ranked_idx]

            # Hit Rate @ K:
            # whether the true positive appears in the top-K ranked results
            hr = 1.0 if pos_item in ranked_items[:k] else 0.0

            if pos_item in ranked_items[:k]:
                rank = ranked_items.index(pos_item)

                # NDCG rewards higher positions more strongly.
                ndcg = 1.0 / math.log2(rank + 2)

                # Since there is only one relevant item, AP simplifies to reciprocal rank.
                ap = 1.0 / (rank + 1)
            else:
                ndcg = 0.0
                ap = 0.0

            hr_list.append(hr)
            ndcg_list.append(ndcg)
            map_list.append(ap)

    if len(hr_list) == 0:
        return 0.0, 0.0, 0.0

    return float(np.mean(hr_list)), float(np.mean(ndcg_list)), float(np.mean(map_list))


# ============================================================
# Training loop with model selection based on validation NDCG
# ============================================================
# NDCG@10 is chosen as the early stopping metric because:
# - it measures ranking quality
# - it cares about the position of the positive item
# - it is usually more informative than hit rate alone
best_val_ndcg = -1.0
best_state = None
best_epoch = -1
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_pointwise(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=DEVICE
    )

    val_hr10, val_ndcg10, val_map10 = evaluate_1vs99(
        model=model,
        eval_users=val_eval_users,
        user_one_pos=val_user_one_pos,
        seen_user_items=val_seen_user_items,
        num_items=num_items,
        k=TOPK,
        num_neg=NUM_NEG_EVAL,
        device=DEVICE,
        seed=SEED
    )

    print(
        f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val HR@{TOPK}: {val_hr10:.4f} | "
        f"Val NDCG@{TOPK}: {val_ndcg10:.4f} | "
        f"Val MAP@{TOPK}: {val_map10:.4f}"
    )

    # Save the best model based on validation NDCG@10.
    if val_ndcg10 > best_val_ndcg:
        best_val_ndcg = val_ndcg10
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience_counter = 0
        print("Saved best model.")
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

# Restore the best checkpoint before final test evaluation.
if best_state is not None:
    model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best Val NDCG@{TOPK}: {best_val_ndcg:.4f}")

# Final evaluation on the test split.
# This is done only once after model selection to keep the test estimate unbiased.
test_hr10, test_ndcg10, test_map10 = evaluate_1vs99(
    model=model,
    eval_users=test_eval_users,
    user_one_pos=test_user_one_pos,
    seen_user_items=test_seen_user_items,
    num_items=num_items,
    k=TOPK,
    num_neg=NUM_NEG_EVAL,
    device=DEVICE,
    seed=SEED
)

print("\nFinal Test Results (Last.fm):")
print(f"Test HR@{TOPK}: {test_hr10:.4f}")
print(f"Test NDCG@{TOPK}: {test_ndcg10:.4f}")
print(f"Test MAP@{TOPK}: {test_map10:.4f}")

Raw shape: (92834, 3)
   userID  artistID  weight
0       2        51   13883
1       2        52   11690
2       2        53   11351
3       2        54   10300
4       2        55    8983

After filtering: (71411, 3)

With pseudo order:
   userID  artistID  weight  pseudo_ts
0       2       100    1315          1
1       2        99    1330          2
2       2        98    1332          3
3       2        97    1337          4
4       2        96    1342          5
5       2        95    1363          6
6       2        93    1407          7
7       2        91    1438          8
8       2        90    1471          9
9       2        89    1519         10

Train: (67667, 4)
Val: (1865, 4)
Test: (1865, 4)

num_users: 1865
num_items: 2828

Encoded train: (67667, 7)
Encoded val: (1865, 7)
Encoded test: (1865, 7)

val_eval_users: 1865
test_eval_users: 1865

Sample batch shapes:
user: torch.Size([2048])
items: torch.Size([2048, 31])
labels: torch.Size([2048, 31])

Model:
TwoTowerImplici